In [1]:
from pathlib import Path
import pandas as pd
import re


import warnings
warnings.filterwarnings('ignore')

import os
import time
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedShuffleSplit, StratifiedKFold, GridSearchCV
from sklearn.metrics import (accuracy_score, f1_score, confusion_matrix,
                             roc_auc_score, precision_recall_curve, ConfusionMatrixDisplay, balanced_accuracy_score)
import joblib


import joblib, os
import numpy as np
import pandas as pd
import torch

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau

import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
import shap
import phate
from scipy.stats import spearmanr

# Set reproducibility seeds
np.random.seed(42)
torch.manual_seed(42)
import random
random.seed(42)

In [2]:
# Load the raw tables
DATA_DIR = Path("/def-singhm/masoudk")
MUT_INFO_PATH     = 'mut_info.tsv'
RNASEQ_PATH = 'rnaseq_data.tsv'
META_PATH = 'meta_data.tsv'

# Load metadata normally
meta_df = pd.read_csv(META_PATH, sep="\t")

# Read first row for gene names
header_df = pd.read_csv(RNASEQ_PATH, sep="\t", header=None, nrows=1, dtype=str)
gene_names = header_df.iloc[0, :].tolist()

# Read data rows (skip header)
rnaseq_df = pd.read_csv(RNASEQ_PATH, sep="\t", header=None, skiprows=1, dtype=str, engine='python', on_bad_lines='warn')

# Pad/truncate to match header length
if rnaseq_df.shape[1] > len(gene_names):
    rnaseq_df = rnaseq_df.iloc[:, :len(gene_names)]
elif rnaseq_df.shape[1] < len(gene_names):
    for i in range(len(gene_names) - rnaseq_df.shape[1]):
        rnaseq_df[len(rnaseq_df.columns)] = None

rnaseq_df.columns = gene_names

# Rename first column to "sample_id"
rnaseq_df = rnaseq_df.rename(columns={gene_names[0]: 'sample_id'})

print(f"RNA-seq shape: {rnaseq_df.shape}")

RNA-seq shape: (6606, 19311)


In [3]:
mut_cols = pd.read_csv(MUT_INFO_PATH, sep='\t', nrows=0).columns.astype(str).tolist()
[c for c in mut_cols if 'tp53' in c.lower()]

['gene.TP53', 'gene.TP53BP1']

In [4]:
TCGA_ID_PATTERN = re.compile(r"TCGA-[A-Z0-9]+-[A-Z0-9]+-\d+")


def _clean_meta(df: pd.DataFrame) -> pd.DataFrame:
    meta = df.copy()
    meta.columns = (
        meta.columns.astype(str)
        .str.strip()
        .str.strip('"')
        .str.lower()
        .str.replace(" ", "_", regex=False)
    )
    
    if 'sample' in meta.columns:
        meta['patient_id'] = meta['sample'].astype(str).str.strip().str.replace('"', '')

    
    meta = meta.dropna(subset=['patient_id']).drop_duplicates('patient_id')

    if "histological_grade" in meta.columns:
        meta["histological_grade"] = (
            meta["histological_grade"].astype(str).str.strip().str.upper().replace({"": pd.NA})
        )

    return meta


def _clean_rnaseq(df: pd.DataFrame) -> pd.DataFrame:
    """
    RNA-seq data structure:
    - Header row: gene names (19311 fields)
    - Data rows: TCGA sample ID in col 1, then 19310 expression values
    """
    # Extract gene names from header
    gene_names = df.columns.astype(str).str.strip().str.replace('"', '')
    
    # Extract sample IDs from first column (no normalization)
    sample_ids = df.iloc[:, 0].astype(str).str.strip().str.replace('"', '')
    
    # Extract expression data: columns 2 onwards
    expr_data = df.iloc[:, 1:].copy()
    expr_data.columns = gene_names[:len(expr_data.columns)]
    expr_data = expr_data.apply(pd.to_numeric, errors="coerce")
    
    # Create dataframe with sample_id and gene expressions
    rna_clean = pd.concat([
        sample_ids.reset_index(drop=True),
        expr_data.reset_index(drop=True)
    ], axis=1)
    
    rna_clean.columns = ['patient_id'] + list(expr_data.columns)
    
    # Remove any empty sample IDs
    rna_clean = rna_clean[rna_clean['patient_id'].str.len() > 0].dropna(subset=['patient_id'])
    
    rna_clean = rna_clean.set_index('patient_id')
    
    # Handle duplicate patients by averaging
    rna_final = rna_clean.groupby(level=0).mean()
    rna_final = rna_final.T  # genes x patients
    rna_final['gene_id'] = rna_final.index
    rna_final = rna_final[['gene_id'] + [c for c in rna_final.columns if c != 'gene_id']]
    rna_final = rna_final.reset_index(drop=True)
    
    return rna_final


clean_meta_df = _clean_meta(meta_df)
clean_rnaseq_df = _clean_rnaseq(rnaseq_df)

rnaseq_patient_ids = set(clean_rnaseq_df.columns) - {"gene_id"}
matched_patient_ids = sorted(set(clean_meta_df["patient_id"]) & rnaseq_patient_ids)

clean_meta_df = clean_meta_df[clean_meta_df["patient_id"].isin(matched_patient_ids)].reset_index(drop=True)
clean_rnaseq_df = clean_rnaseq_df[["gene_id"] + matched_patient_ids]

print(f"Matched {len(matched_patient_ids)} patients across both tables")
print(f"Metadata: {len(clean_meta_df)} rows")
print(f"RNA-seq: {len(clean_rnaseq_df)} genes × {len(matched_patient_ids)} samples")
clean_meta_df.head()

Matched 6606 patients across both tables
Metadata: 6606 rows
RNA-seq: 19310 genes × 6606 samples


,meta_data.tsv,sample_type,project_id,rnaseqid,mutid,sample,x_patient,cancer.type.abbreviation,age_at_initial_pathologic_diagnosis,gender,...,os.time,dss,dss.time,dfi,dfi.time,pfi,pfi.time,redaction,x_primary_disease,patient_id
0,1.0,Primary Tumor,TCGA-GBM,TCGA-02-0047-01A,TCGA-02-0047-01,TCGA-02-0047-01,TCGA-02-0047,GBM,78.0,MALE,...,448.0,1.0,448.0,NaN,NaN,1.0,57.0,NaN,glioblastoma multiforme,TCGA-02-0047-01
1,1.0,Primary Tumor,TCGA-GBM,TCGA-02-0055-01A,TCGA-02-0055-01,TCGA-02-0055-01,TCGA-02-0055,GBM,62.0,FEMALE,...,76.0,1.0,76.0,NaN,NaN,1.0,6.0,NaN,glioblastoma multiforme,TCGA-02-0055-01
2,1.0,Primary Tumor,TCGA-GBM,TCGA-02-2483-01A,TCGA-02-2483-01,TCGA-02-2483-01,TCGA-02-2483,GBM,43.0,MALE,...,466.0,0.0,466.0,NaN,NaN,0.0,466.0,NaN,glioblastoma multiforme,TCGA-02-2483-01
3,1.0,Primary Tumor,TCGA-GBM,TCGA-02-2485-01A,TCGA-02-2485-01,TCGA-02-2485-01,TCGA-02-2485,GBM,53.0,MALE,...,470.0,0.0,470.0,NaN,NaN,1.0,186.0,NaN,glioblastoma multiforme,TCGA-02-2485-01
4,1.0,Primary Tumor,TCGA-GBM,TCGA-02-2486-01A,TCGA-02-2486-01,TCGA-02-2486-01,TCGA-02-2486,GBM,64.0,MALE,...,618.0,1.0,618.0,NaN,NaN,1.0,618.0,NaN,glioblastoma multiforme,TCGA-02-2486-01


In [5]:
meta_test = meta_df.copy()
cols_lower = (
    meta_test.columns.astype(str)
    .str.strip()
    .str.strip('"')
    .str.lower()
    .str.replace(" ", "_", regex=False)
)
meta_test.columns = cols_lower

id_candidates = ['sample', 'x_patient', 'rnaseqid', 'sample_type_id', 'mutid']

def _normalize_patient_id(val):
    if pd.isna(val):
        return pd.NA
    s = str(val).strip().replace('"', '')
    return s if s else pd.NA

def _select_patient_id_test(row):
    for col in id_candidates:
        val = row.get(col)
        normalized = _normalize_patient_id(val)
        print(f"    {col}: {val} -> {normalized}")
        if not pd.isna(normalized):
            return normalized
    return pd.NA

In [6]:
print("Meta patient_id):")
print(clean_meta_df['patient_id'].head().tolist())

print("\nRNA-seq patient_id:")
print(list(rnaseq_patient_ids)[:5])

print(f"\nMeta unique count: {len(set(clean_meta_df['patient_id']))}")
print(f"RNA-seq unique count: {len(rnaseq_patient_ids)}")

print("\nIntersection:")
intersection = set(clean_meta_df["patient_id"]) & rnaseq_patient_ids
print(f"Matched: {len(intersection)}")

Meta patient_id):
['TCGA-02-0047-01', 'TCGA-02-0055-01', 'TCGA-02-2483-01', 'TCGA-02-2485-01', 'TCGA-02-2486-01']

RNA-seq patient_id:
['TCGA-CV-7438-01', 'TCGA-24-1413-01', 'TCGA-52-7809-01', 'TCGA-EJ-A65F-01', 'TCGA-C5-A1BJ-01']

Meta unique count: 6606
RNA-seq unique count: 6606

Intersection:
Matched: 6606


In [7]:
print("=== DATASET BEFORE LABELING ===")
print(f"Metadata shape: {clean_meta_df.shape}")
print(f"RNA-seq shape: {clean_rnaseq_df.shape}")

print(f"Loading mutation table: {MUT_INFO_PATH}")
mut_df = pd.read_csv(MUT_INFO_PATH, sep='\t', dtype=str)

mut_df.columns = (
    mut_df.columns.astype(str)
    .str.strip()
    .str.strip('"')
    .str.lower()
    .str.replace(' ', '_', regex=False)
)

index_ids = pd.Index(mut_df.index).astype(str)
index_tcga_fraction = index_ids.str.match(r'^TCGA-[A-Z0-9]+-[A-Z0-9]+-\d+').mean()

if index_tcga_fraction > 0.5:
    mut_df = mut_df.copy()
    mut_df['patient_id'] = index_ids.str.strip().str.replace('"', '')
    id_col = 'patient_id'
else:
    id_candidates = ['patient_id', 'sample', 'x_patient', 'rnaseqid', 'sample_id', 'mutid']
    id_col = next((c for c in id_candidates if c in mut_df.columns), mut_df.columns[0])
    mut_df[id_col] = mut_df[id_col].astype(str).str.strip().str.replace('"', '')

mut_df = mut_df.dropna(subset=[id_col]).drop_duplicates(id_col)

mutation_cols = [c for c in mut_df.columns if c != id_col]
tp53_col = next((c for c in mutation_cols if 'tp53' in c.lower()), None)

print(f"Found TP53 column: {tp53_col}")

binary_mut_info = mut_df[[id_col, tp53_col]].copy()
binary_mut_info['mutation_label'] = binary_mut_info[tp53_col].apply(
    lambda x: 0 if str(x).strip().lower() == 'wild' else 1
)

mutation_by_patient = binary_mut_info.set_index(id_col)['mutation_label']
clean_meta_df['mutation_label'] = clean_meta_df['patient_id'].map(mutation_by_patient)

valid_idx = clean_meta_df['mutation_label'].notna()
labeled_meta_df = clean_meta_df[valid_idx].copy().reset_index(drop=True)
labeled_meta_df['mutation_label'] = labeled_meta_df['mutation_label'].astype(int)

valid_patient_ids = set(labeled_meta_df['patient_id'])
labeled_rnaseq_df = clean_rnaseq_df[['gene_id'] + sorted([p for p in clean_rnaseq_df.columns if p in valid_patient_ids])].reset_index(drop=True)

print("\n=== DATASET AFTER LABELING ===")
print(f"Metadata shape: {labeled_meta_df.shape}")
print(f"RNA-seq shape: {labeled_rnaseq_df.shape}")
print("\nMutation label distribution:")
print(labeled_meta_df['mutation_label'].value_counts().sort_index())
print(f"  0 (Wild-type): {(labeled_meta_df['mutation_label'] == 0).sum()}")
print(f"  1 (Mutated):   {(labeled_meta_df['mutation_label'] == 1).sum()}")


=== DATASET BEFORE LABELING ===
Metadata shape: (6606, 41)
RNA-seq shape: (19310, 6607)
Loading mutation table: mut_info.tsv
Found TP53 column: gene.tp53

=== DATASET AFTER LABELING ===
Metadata shape: (6606, 42)
RNA-seq shape: (19310, 6607)

Mutation label distribution:
mutation_label
0    4063
1    2543
Name: count, dtype: int64
  0 (Wild-type): 4063
  1 (Mutated):   2543


In [8]:
def stage_0_prepare_data(X, y, test_size=0.15, val_size=0.15, random_state=42, output_dir='deployment'):
    os.makedirs(output_dir, exist_ok=True)

    # Stratified split: train+val vs test
    splitter1 = StratifiedShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
    train_val_idx, test_idx = next(splitter1.split(X, y))

    X_train_val = X[train_val_idx]
    y_train_val = y[train_val_idx]
    X_test = X[test_idx]
    y_test = y[test_idx]

    # Stratified split: train vs val
    val_size_adjusted = val_size / (1.0 - test_size)
    splitter2 = StratifiedShuffleSplit(n_splits=1, test_size=val_size_adjusted, random_state=random_state)
    train_idx, val_idx = next(splitter2.split(X_train_val, y_train_val))

    X_train = X_train_val[train_idx]
    y_train = y_train_val[train_idx]
    X_val = X_train_val[val_idx]
    y_val = y_train_val[val_idx]

    # Fit scaler on train, apply to all
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)

    # Save scaler
    scaler_path = os.path.join(output_dir, 'mutation_scaler.pkl')
    joblib.dump(scaler, scaler_path)

    result = {
        'X_train': X_train_scaled,
        'y_train': y_train,
        'X_val': X_val_scaled,
        'y_val': y_val,
        'X_test': X_test_scaled,
        'y_test': y_test,
        'scaler': scaler,
        'scaler_path': scaler_path,
        'n_features': X.shape[1],
    }

    print(f"STAGE 0: Data Prepared")
    print(f"  Train: {result['X_train'].shape} | Val: {result['X_val'].shape} | Test: {result['X_test'].shape}")
    print(f"  Label distribution:")
    print(f"    Train - Wild-type: {(y_train==0).sum()}, Mutated: {(y_train==1).sum()}")
    print(f"    Val   - Wild-type: {(y_val==0).sum()}, Mutated: {(y_val==1).sum()}")
    print(f"    Test  - Wild-type: {(y_test==0).sum()}, Mutated: {(y_test==1).sum()}")

    return result

X_raw = labeled_rnaseq_df.drop('gene_id', axis=1).T.values  # (samples x genes)
y_binary = labeled_meta_df['mutation_label'].values

data = stage_0_prepare_data(X_raw, y_binary, output_dir='deployment')


STAGE 0: Data Prepared
  Train: (4624, 19310) | Val: (991, 19310) | Test: (991, 19310)
  Label distribution:
    Train - Wild-type: 2844, Mutated: 1780
    Val   - Wild-type: 609, Mutated: 382
    Test  - Wild-type: 610, Mutated: 381


In [9]:
# convert to numeric
for k in ['y_train', 'y_val', 'y_test']:
    data[k] = pd.to_numeric(data[k], errors='coerce').astype(int)

for k in ['X_train', 'X_val', 'X_test']:
    data[k] = np.nan_to_num(np.asarray(data[k], dtype=np.float32), nan=0.0, posinf=0.0, neginf=0.0)

print("Train classes:", np.unique(data['y_train'], return_counts=True))
print("Val classes:", np.unique(data['y_val'], return_counts=True))
print("Test classes:", np.unique(data['y_test'], return_counts=True))

y_train = data['y_train']

# class imbalance methods
n_pos = int((y_train == 1).sum())
n_neg = int((y_train == 0).sum())
scale_pos_weight = float(n_neg) / float(max(n_pos, 1))
n_samples = len(y_train)
w_neg = n_samples / (2.0 * max(n_neg, 1))
w_pos = n_samples / (2.0 * max(n_pos, 1))
sample_weight = np.where(y_train == 1, w_pos, w_neg)

print(f"  Class balance (train): neg={n_neg}, pos={n_pos}")
print(f"  Imbalance settings: scale_pos_weight={scale_pos_weight:.4f}, w_neg={w_neg:.4f}, w_pos={w_pos:.4f}")


Train classes: (array([0, 1]), array([2844, 1780]))
Val classes: (array([0, 1]), array([609, 382]))
Test classes: (array([0, 1]), array([610, 381]))
  Class balance (train): neg=2844, pos=1780
  Imbalance settings: scale_pos_weight=1.5978, w_neg=0.8129, w_pos=1.2989


In [10]:
# Random Forest Baseline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score, classification_report

X_train, y_train = data['X_train'], data['y_train']
X_val, y_val = data['X_val'], data['y_val']
X_test, y_test = data['X_test'], data['y_test']

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train, sample_weight=sample_weight)

# Validation
rf_val_pred = rf_model.predict(X_val)
rf_val_prob = rf_model.predict_proba(X_val)[:, 1]

print("=== Random Forest: VALIDATION ===")
print("Accuracy:", round(accuracy_score(y_val, rf_val_pred), 4))
print("Balanced Accuracy:", round(balanced_accuracy_score(y_val, rf_val_pred), 4))
print("F1:", round(f1_score(y_val, rf_val_pred), 4))
print("ROC AUC:", round(roc_auc_score(y_val, rf_val_prob), 4))

# Test
rf_test_pred = rf_model.predict(X_test)
rf_test_prob = rf_model.predict_proba(X_test)[:, 1]

print("\n=== Random Forest: TEST ===")
print("Accuracy:", round(accuracy_score(y_test, rf_test_pred), 4))
print("Balanced Accuracy:", round(balanced_accuracy_score(y_test, rf_test_pred), 4))
print("F1:", round(f1_score(y_test, rf_test_pred), 4))
print("ROC AUC:", round(roc_auc_score(y_test, rf_test_prob), 4))
print("\nClassification Report:\n")
print(classification_report(y_test, rf_test_pred, digits=4))


=== Random Forest: VALIDATION ===
Accuracy: 0.8143
Balanced Accuracy: 0.7875
F1: 0.7356
ROC AUC: 0.9004

=== Random Forest: TEST ===
Accuracy: 0.8123
Balanced Accuracy: 0.7899
F1: 0.7395
ROC AUC: 0.9044

Classification Report:

              precision    recall  f1-score   support

           0     0.8222    0.8869    0.8533       610
           1     0.7928    0.6929    0.7395       381

    accuracy                         0.8123       991
   macro avg     0.8075    0.7899    0.7964       991
weighted avg     0.8109    0.8123    0.8096       991



In [12]:
# linear SVC Baseline 
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score, classification_report

X_train, y_train = data['X_train'], data['y_train']
X_val, y_val = data['X_val'], data['y_val']
X_test, y_test = data['X_test'], data['y_test']

svm_model = LinearSVC(
    C=1.0,
    random_state=42,
    max_iter=10000
)

svm_model.fit(X_train, y_train, sample_weight=sample_weight)

# Validation
svm_val_pred = svm_model.predict(X_val)
svm_val_score = svm_model.decision_function(X_val)

print("=== Linear SVM: Validation ===")
print("Accuracy:", round(accuracy_score(y_val, svm_val_pred), 4))
print("Balanced Accuracy:", round(balanced_accuracy_score(y_val, svm_val_pred), 4))
print("F1:", round(f1_score(y_val, svm_val_pred), 4))
print("ROC AUC:", round(roc_auc_score(y_val, svm_val_score), 4))

# Test
svm_test_pred = svm_model.predict(X_test)
svm_test_score = svm_model.decision_function(X_test)

print("\n=== Linear SVM: Test ===")
print("Accuracy:", round(accuracy_score(y_test, svm_test_pred), 4))
print("Balanced Accuracy:", round(balanced_accuracy_score(y_test, svm_test_pred), 4))
print("F1:", round(f1_score(y_test, svm_test_pred), 4))
print("ROC AUC:", round(roc_auc_score(y_test, svm_test_score), 4))
print("\nClassification Report:\n")
print(classification_report(y_test, svm_test_pred, digits=4))

=== Linear SVM: Validation ===
Accuracy: 0.8527
Balanced Accuracy: 0.844
F1: 0.8084
ROC AUC: 0.9122

=== Linear SVM: Test ===
Accuracy: 0.8335
Balanced Accuracy: 0.8253
F1: 0.7849
ROC AUC: 0.8895

Classification Report:

              precision    recall  f1-score   support

           0     0.8678    0.8607    0.8642       610
           1     0.7798    0.7900    0.7849       381

    accuracy                         0.8335       991
   macro avg     0.8238    0.8253    0.8245       991
weighted avg     0.8339    0.8335    0.8337       991



In [11]:
# XGBoost Baseline 
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score, classification_report
import joblib
import os

X_train, y_train = data['X_train'], data['y_train']
X_val, y_val = data['X_val'], data['y_val']
X_test, y_test = data['X_test'], data['y_test']

xgb_model = xgb.XGBClassifier(
        n_estimators=900,
        max_depth=20,
        learning_rate=0.07,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        max_delta_step=3,
        random_state=42,
        n_jobs=-1,
        eval_metric='logloss',
        verbose=0
    )

xgb_model.fit(X_train, y_train)

# Validation
xgb_val_pred = xgb_model.predict(X_val)
xgb_val_prob = xgb_model.predict_proba(X_val)[:, 1]

print("=== XGBOOST: Validation ===")
print("Accuracy:", round(accuracy_score(y_val, xgb_val_pred), 4))
print("Balanced Accuracy:", round(balanced_accuracy_score(y_val, xgb_val_pred), 4))
print("F1:", round(f1_score(y_val, xgb_val_pred), 4))
print("ROC AUC:", round(roc_auc_score(y_val, xgb_val_prob), 4))

# Test
xgb_test_pred = xgb_model.predict(X_test)
xgb_test_prob = xgb_model.predict_proba(X_test)[:, 1]

print("\n=== XGBOOST: Test ===")
print("Accuracy:", round(accuracy_score(y_test, xgb_test_pred), 4))
print("Balanced Accuracy:", round(balanced_accuracy_score(y_test, xgb_test_pred), 4))
print("F1:", round(f1_score(y_test, xgb_test_pred), 4))
print("ROC AUC:", round(roc_auc_score(y_test, xgb_test_prob), 4))
print("\nClassification Report:\n")
print(classification_report(y_test, xgb_test_pred, digits=4))

=== XGBOOST: Validation ===
Accuracy: 0.887
Balanced Accuracy: 0.8832
F1: 0.8553
ROC AUC: 0.9369

=== XGBOOST: Test ===
Accuracy: 0.8749
Balanced Accuracy: 0.8742
F1: 0.8426
ROC AUC: 0.931

Classification Report:

              precision    recall  f1-score   support

           0     0.9161    0.8770    0.8961       610
           1     0.8157    0.8714    0.8426       381

    accuracy                         0.8749       991
   macro avg     0.8659    0.8742    0.8694       991
weighted avg     0.8775    0.8749    0.8756       991

